# [6.3] Transcoders and Attribution Graphs - Solutions

Reference validation notebook for the section-local transcoder implementation. This executes the visible tests against `solutions.py`, checks the CPU notebook contract, and verifies the committed CUDA report highlights.

Expected CUDA highlights: pinned TransformerLens `gelu-1l` loads on CUDA, exact MLP-feature oracle replacement matches model logits, a tiny ReLU transcoder beats the zero-output baseline on held-out activations, top-token agreement clears the threshold, and the top-feature attribution graph preserves/damages the target logit difference more than low-effect controls.


In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter6_sparse_feature_methods"
section = "part3_transcoders_attribution_graphs"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_transcoders_attribution_graphs.tests as tests
from part3_transcoders_attribution_graphs import solutions


In [ ]:
tests.test_transcoder_forward_matches_reference_and_relu_rules(
    solutions.transcoder_forward,
)
tests.test_target_logit_diff_and_replacement_report_match_reference(
    solutions.target_logit_diff,
    solutions.transcoder_replacement_report,
)
tests.test_feature_logit_contributions_reduce_all_nonfeature_dimensions(
    solutions.feature_logit_contributions,
)
tests.test_build_attribution_edges_keeps_top_input_and_logit_edges(
    solutions.build_attribution_edges,
    solutions.graph_reproducible,
)
tests.test_graph_reproducible_rejects_structure_and_weight_changes(
    solutions.AttributionEdge,
    solutions.graph_reproducible,
)
tests.test_attribution_graph_report_preservation_and_damage_controls(
    solutions.graph_density,
    solutions.attribution_graph_report,
)
tests.test_notebook_contract(solutions.run_smoke_test)


In [ ]:
contract = solutions.run_smoke_test(cpu=True)
contract


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert gpu["preflight_passed"], "6.3 CUDA preflight should pass."
assert gpu["model_name"] == "gelu-1l", "The real-model path should use pinned TransformerLens gelu-1l."
assert gpu["hf_revision"] == "bddc0e332f0ae84279e6a6a45d91b314899e1603", "gelu-1l revision should remain pinned."
assert gpu["oracle_mlp_out_max_abs_error"] <= 1e-5, "Oracle MLP output reconstruction should be exact to tolerance."
assert gpu["oracle_logits_max_abs_error"] <= 5e-5, "Oracle replacement logits should match model logits to tolerance."
assert gpu["oracle_preserves_logit_diff"], "Oracle replacement should preserve the target logit diff."
assert gpu["trained_transcoder_heldout_mse_ratio"] <= 0.5, "Trained transcoder should beat the zero-output baseline on held-out activations."
assert gpu["trained_replacement_top1_agreement"] >= 0.75, "Trained replacement logits should preserve top-token behavior on most positions."
assert gpu["graph_feature_count"] == 64, "The graph report should use the locked top-64 feature set."
assert gpu["graph_preserves_logit_diff"], "Top graph features should preserve the target logit diff."
assert gpu["graph_passes_damage_control"], "Top graph feature removal should damage behavior more than low-effect controls."
assert gpu["graph_reproducible"], "Repeated graph construction should be reproducible."
assert gpu["peak_vram_gb"] <= 1.0, "The 6.3 preflight should stay under the locked 1GB budget."
{
    "oracle_logits_max_abs_error": gpu["oracle_logits_max_abs_error"],
    "trained_transcoder_heldout_mse_ratio": gpu["trained_transcoder_heldout_mse_ratio"],
    "trained_replacement_top1_agreement": gpu["trained_replacement_top1_agreement"],
    "graph_topk_damage": gpu["graph_topk_damage"],
    "graph_random_damage": gpu["graph_random_damage"],
    "peak_vram_gb": gpu["peak_vram_gb"],
}
